# BAI-01 - Train the PPE detector on Google Colab (T4)

The laptop does **not** train. Everything heavy happens on Colab's free T4 GPU.

**Before you start**
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Upload `colab_bundle.zip` (project code + the 1143-image dataset, ~69 MB) from the project
   folder to Google Drive at `MyDrive/ppe_project/colab_bundle.zip`.
3. Run the cells top to bottom.

Everything is saved to `MyDrive/ppe_project/`, so a Colab disconnect costs nothing: re-run
cells 1-4 and then the **RESUME** cell.

| Cell | What it does | Time |
|---|---|---|
| 1 | Check the GPU | seconds |
| 2 | Mount Google Drive | seconds |
| 3 | Unpack the bundle + install Ultralytics | 1-2 min |
| 4 | Validate the dataset | seconds |
| 5 | Sanity run (3 epochs, 10 % of the data) | 1 min |
| 6 | Full training (30 epochs) | ~10 min |
| 7 | RESUME after a disconnect | - |
| 8 | Metrics on the VALIDATION split | 1 min |
| 9 | Metrics on the TEST split (run once) | 1 min |
| 10 | Export best.pt + reports, download them | 1 min |


## 1 - Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then Runtime -> Restart"


## 2 - Mount Google Drive (everything important is saved here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/ppe_project'
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Saving to:", DRIVE_DIR)


## 3 - Unpack the project bundle + install Ultralytics

`colab_bundle.zip` was built on the laptop and contains `training/`, `scripts/` and the prepared
dataset `datasets/ppe4/` (1000 train / 84 val / 59 test images). Upload it to
`MyDrive/ppe_project/` first - no GitHub account and no Kaggle token are needed.

The dataset is unpacked to Colab's **local disk** (`/content`), not to Drive, because reading
thousands of small files over Drive is slow.


In [ ]:
BUNDLE = f"{DRIVE_DIR}/colab_bundle.zip"
PROJECT = "/content/ppe-safety-copilot"

import os, shutil
assert os.path.exists(BUNDLE), f"Not found: {BUNDLE}  -> upload colab_bundle.zip to MyDrive/ppe_project/"
if os.path.exists(PROJECT):
    shutil.rmtree(PROJECT)
os.makedirs(PROJECT, exist_ok=True)
!unzip -q -o "$BUNDLE" -d $PROJECT
%cd $PROJECT
!mkdir -p models docs/evaluation
!pip install -q "ultralytics>=8.4.0" pyyaml pillow
import ultralytics; print("Ultralytics", ultralytics.__version__)


## 4 - Check the dataset before spending GPU time on it

In [ ]:
!python -m scripts.check_dataset --data datasets/ppe4


## 5 - Sanity run: 3 epochs on 10 % of the data

If this finishes and the mAP moves at all, the full run will work too.


In [ ]:
!python -m training.train --data datasets/ppe4/data.yaml --epochs 3 --fraction 0.1 --batch 16 --workers 2 --device 0 --project "$DRIVE_DIR/runs" --name smoke


## 6 - Full training

The T4 has 16 GB of VRAM, so `--batch 16` is comfortable here (the laptop would use 8).
Settings match `docs/decisions.md` D-011: 30 epochs, imgsz 640, patience 8, seed 42, and
**mosaic off** - our training images are already Roboflow mosaics (D-010a).

Keep this tab open. Checkpoints are written to Drive after every epoch, so a disconnect only
costs the current epoch.


In [ ]:
RUN_NAME = "ppe4_yolo26n"
!python -m training.train --data datasets/ppe4/data.yaml --epochs 30 --imgsz 640 --batch 16 --workers 2 --device 0 --patience 8 --mosaic 0.0 --project "$DRIVE_DIR/runs" --name $RUN_NAME
!cp models/*_info.json "$DRIVE_DIR/" 2>/dev/null; echo done


## 7 - RESUME (only after a disconnect: run cells 1-4 first, then this)

In [ ]:
RUN_NAME = "ppe4_yolo26n"
!python -m training.train --resume --project "$DRIVE_DIR/runs" --name $RUN_NAME
!cp models/*_info.json "$DRIVE_DIR/" 2>/dev/null; echo done


## 8 - Metrics on the VALIDATION split

This is the split you are allowed to look at while tuning.


In [ ]:
RUN_NAME = "ppe4_yolo26n"
BEST = f"{DRIVE_DIR}/runs/{RUN_NAME}/weights/best.pt"
!python -m training.validate --weights "$BEST" --data datasets/ppe4/data.yaml --split val --batch 16 --device 0
!cp docs/evaluation/detector_val_metrics.json "$DRIVE_DIR/"


## 9 - Metrics on the TEST split - run ONCE, when all decisions are final

The test images were never trained on or tuned against, so this is the honest number for the
report. Quote it together with the per-class instance counts: the test split has only 59 images.


In [ ]:
RUN_NAME = "ppe4_yolo26n"
BEST = f"{DRIVE_DIR}/runs/{RUN_NAME}/weights/best.pt"
!python -m training.validate --weights "$BEST" --data datasets/ppe4/data.yaml --split test --batch 16 --device 0
!cp docs/evaluation/detector_test_metrics.json "$DRIVE_DIR/"


## 10 - Export: best model + reports, then download

Put the downloaded files on the laptop:
* `ppe4_yolo26n_best.pt` -> `models\\`
* `detector_*_metrics.json`, `results.csv`, `results.png`, `confusion_matrix.png` -> `docs\\evaluation\\`

Then on the laptop: `python -m edge.run_edge --weights models\\ppe4_yolo26n_best.pt`


In [ ]:
import shutil, os, glob
RUN_NAME = "ppe4_yolo26n"
run_dir = f"{DRIVE_DIR}/runs/{RUN_NAME}"
export = f"{DRIVE_DIR}/export"
os.makedirs(export, exist_ok=True)

shutil.copy2(f"{run_dir}/weights/best.pt", f"{export}/{RUN_NAME}_best.pt")
for f in (glob.glob(f"{DRIVE_DIR}/*_info.json") + glob.glob(f"{DRIVE_DIR}/detector_*_metrics.json")
          + [f"{run_dir}/results.csv", f"{run_dir}/results.png", f"{run_dir}/confusion_matrix.png",
             f"{run_dir}/args.yaml"]):
    if os.path.exists(f):
        shutil.copy2(f, export)
print("Exported to Drive:", sorted(os.listdir(export)))

shutil.make_archive("/content/ppe4_results", "zip", export)
from google.colab import files
files.download("/content/ppe4_results.zip")
